# Day05 下午个人项目：电商用户多维分析

**姓名：** 24012453  
**专题方向：** A / B / C / D / E

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "24012453"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

In [ ]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [ ]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

In [ ]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode",
]


# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = pd.Series({
    "行数": len(df), "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}, name="验收结果")

display(validation.to_frame())


# TODO 3：展示验收结果
# display(validation)

In [ ]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

### 数据粒度说明

请用一句话说明一行数据代表什么：

> 一行数据代表一名电商用户（CustomerID），不是一笔订单。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> CustomerID 是用户标识符而非业务指标，其数值大小无业务含义。对标识符求平均、求和等操作没有统计意义，若当做数值特征参与建模，模型可能把 ID 大小误解为业务规律。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [ ]:
# TODO：计算公共基础指标

overall_metrics = pd.DataFrame({
    "指标": ["用户数", "流失人数", "流失率", "平均订单数", "订单数中位数",
             "平均优惠券数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数"],
    "数值": [
        df["CustomerID"].nunique(), df["Churn"].sum(), df["Churn"].mean(),
        df["OrderCount"].mean(), df["OrderCount"].median(),
        df["CouponUsed"].mean(), df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(), df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})

display(overall_metrics)


# TODO：展示结果
# display(overall_metrics)

In [ ]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> 当前样本共有 5630 名用户，总体流失率为 16.84%，约每 6 名用户中有 1 名流失。，例如“当前样本共有……名用户，总体流失率为……”。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [ ]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "TenureGroup"


# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = (
    df.groupby(segment_field, observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
      .sort_values("用户数", ascending=False)
)

display(segment_analysis)


# TODO 3：重置索引、按业务意义排序并展示
# display(segment_analysis)

In [ ]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

### 单维专题分析记录

**本专题要回答的业务问题：**

> 不同生命周期（TenureGroup）用户的流失率和订单行为有何差异？

**数据现象：**

> 0-5年用户（N=2108）流失率 38.2%，订单数均值 2.87；10年以上用户流失率降至 0-6%，订单数均值 4.5+。、用户数、指标和具体数值。

**可能解释：**

> 使用时长与流失率呈负相关，可能老用户已形成使用习惯或已通过忠诚度筛选；需验证新用户流失是否与 onboarding 体验相关。“相关、可能、值得关注、需验证”等有边界语言。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [ ]:
# TODO 1：选择两个不同维度
allowed_cross_fields = ["TenureGroup", "Complain", "PreferedOrderCat", "CityTier", "PreferredLoginDevice", "PreferredPaymentMode"]
dim_1 = "TenureGroup"
dim_2 = "Complain"


cross_analysis = (
    df.groupby([dim_1, dim_2], observed=True)
      .agg(用户数=("CustomerID", "nunique"),
           流失人数=("Churn", "sum"),
           流失率=("Churn", "mean"),
           平均订单数=("OrderCount", "mean"))
      .reset_index()
)
cross_analysis["投诉状态"] = cross_analysis[dim_2].map({0: "无投诉", 1: "有投诉"})
cross_analysis["样本提示"] = np.where(cross_analysis["用户数"] < 30, "小样本", "可观察")
cross_analysis = cross_analysis.sort_values(["流失率", "用户数"], ascending=[False, False])

display(cross_analysis)

In [ ]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

### 双维分析记录

**最值得关注的维度组合：**

> 0-5年 + 有投诉（Complain=1）：用户数较小但流失率极高（62%），值得重点关注。

**该组合的用户数、流失率和比较对象：**

> 0-5年有投诉用户的流失率（62%）远高于同组无投诉用户（26%）和整体流失率（16.84%），投诉行为叠加低使用时长是强风险信号。

**是否存在小样本风险：**

> 有投诉用户总量较少，部分 TenureGroup 组合的样本量低于 30，标记为"小样本"后需谨慎引用。

**为什么不能直接写成因果结论：**

> 投诉与流失之间可能存在双向影响——流失用户可能因不满而投诉，也可能因投诉被无视而流失。当前数据只能展示关联，不能证明"投诉导致流失"。

## 任务5：输出统计报表（必做）

In [ ]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

In [ ]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

## 任务6：结论、限制与建议（必做）

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> 建议：针对新用户（0-5年）推出 onboarding 引导计划，在注册后前 30 天推送优惠券和使用教程，降低早期流失。
> 验证方式：需要 A/B 实验数据，对比接受引导计划的新用户与对照组在 3 个月后的留存率差异；同时追踪投诉用户的处理响应时间和满意度评分变化，评估客服干预对挽回流失的效果。


## 拓展任务（选做）

In [ ]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）

## 最终检查：GitHub提交前验收

In [ ]:
required_files = [
    ROOT / "day5" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？